In [1]:
import os
import json
import asyncio
from typing import List, Optional, Any
from datetime import datetime
from IPython.display import Markdown, display

from langgraph.types import Command
from langchain_core.runnables import RunnableConfig
from langchain.chat_models import init_chat_model
from langchain_core.tools import tool
from langchain_core.messages import AIMessage, ToolMessage
from pydantic import BaseModel, Field
from langgraph.graph import StateGraph, END, MessagesState

In [2]:
from dotenv import load_dotenv
load_dotenv('.env')

True

In [3]:
from langchain_core.rate_limiters import InMemoryRateLimiter

rate_limiter = InMemoryRateLimiter(
    requests_per_second=1,  
    check_every_n_seconds=0.1,
    max_bucket_size=10,
)

mini_llm = init_chat_model("openai:gpt-5-mini", rate_limiter=rate_limiter, max_retries = 5)
adv_llm = init_chat_model("openai:gpt-5", rate_limiter=rate_limiter, max_retries = 5)

In [4]:
#used in Plan which is used in State
class Step(BaseModel):
    title: str
    description: str = Field(..., description="Specify exactly what data to collect")
    execution_res: Optional[str] = Field(
        default=None, description="The Step execution result"
    )

#used in the State object
class Plan(BaseModel):
    thought: str
    title: str
    steps: List[Step] = Field(
        default_factory=list,
        description="Research & Processing steps to get more context",
    )


class State(MessagesState):
    """State for our research agents."""
    research_topic: str = ""
    observations: list[str] = []
    queries: list[str] = []
    plan_iterations: int = 0
    step_iteration: int = 0
    current_plan: Plan | str = None
    final_report: str = ""

In [5]:
plan_prompt = """Today's date is {{ date }}. You are a professional research agent. 
Use the tools at your disposal to research the user's question, do not rely on in-built knowledge or other context, be sure to thoroughly research a given topic and provide any caveats or other considerations with your results, including ways the results could be incomplete or misleading.

Information Standards and Success

A successful research plan must meet these requirements:

Comprehensive Coverage -Explore all aspects of the topic to identify associated themes or trends that shed light on the question at hand -Represent multiple viewpoints or perspectives

Detailed Analysis -For key portions of the question or topic, perform detailed analysis, performing research to further explore a specific area -Detailed data points, facts, and statistics are required -Multiple sources are required for in-depth analysis.

Tools available for use in answering the user's question are:
{{ tools }}

You can construct a plan that uses these tools iteratively, receiving results and planning subsequent steps or queries depending on the results.
"""


In [6]:
from langchain_mcp_adapters.client import MultiServerMCPClient
from langchain_core.tools.structured import StructuredTool

client = MultiServerMCPClient(
    {
        "database": {
            "url": "http://localhost:8000/mcp",
            "transport": "streamable_http",
        }
    }
)

tools = await client.get_tools()

In [7]:
from jinja2 import Template

template = Template(plan_prompt)

rendered = template.render(date=datetime.now().strftime('%m/%d/%Y'), tools=tools)


In [8]:
print(rendered)

Today's date is 09/17/2025. You are a professional research agent. 
Use the tools at your disposal to research the user's question, do not rely on in-built knowledge or other context, be sure to thoroughly research a given topic and provide any caveats or other considerations with your results, including ways the results could be incomplete or misleading.

Information Standards and Success

A successful research plan must meet these requirements:

Comprehensive Coverage -Explore all aspects of the topic to identify associated themes or trends that shed light on the question at hand -Represent multiple viewpoints or perspectives

Detailed Analysis -For key portions of the question or topic, perform detailed analysis, performing research to further explore a specific area -Detailed data points, facts, and statistics are required -Multiple sources are required for in-depth analysis.

Tools available for use in answering the user's question are:
[StructuredTool(name='list_tables', descript

In [9]:
from dataclasses import dataclass, field, fields

#used in apply_prompt_templates
@dataclass(kw_only=True)
class Configuration:
    """The configurable fields."""
    max_plan_iterations: int = 1  # Maximum number of plan iterations
    max_step_num: int = 8  # Maximum number of steps in a plan
    mcp_settings: StructuredTool = None  # MCP settings, including dynamic loaded tools

    @classmethod
    def from_runnable_config(
        cls, config: Optional[RunnableConfig] = None
    ) -> "Configuration":
        """Create a Configuration instance from a RunnableConfig."""
        configurable = (
            config["configurable"] if config and "configurable" in config else {}
        )
        values: dict[str, Any] = {
            f.name: os.environ.get(f.name.upper(), configurable.get(f.name))
            for f in fields(cls)
            if f.init
        }
        return cls(**{k: v for k, v in values.items() if v})

In [10]:
question = "How did the MSRP of EVs in Washington State change over the years?"

initial_state = {
        #Setting up State
        "messages": [{"role": "user", "content": question}],
    }

config = {
    #runtime configuration or variables
        "configurable": {
            "thread_id": "default",
            "max_plan_iterations": 1,
            "max_step_num": 8,
            "mcp_settings": tools
        },
        "recursion_limit": 100,
    }

In [11]:
def prepare_prompt(node, config, messages, plan=None):
    with open(f"{node}.md", "r", encoding="utf-8") as file:
        plan_prompt = file.read()
    template = Template(plan_prompt)

    prompt = template.render(date=datetime.now().strftime('%d/%m/%Y'), tools=config.mcp_settings, messages = messages, plan=plan, max_steps=config.max_step_num)
    return [{"role": "system", "content": prompt}] + messages

def planner_node(state: State, config: RunnableConfig):
    configurable = Configuration.from_runnable_config(config)
    messages = state.get("messages")
    plan_llm = adv_llm.with_structured_output(Plan,method="json_mode")
    #if plan_iterations >= configurable.max_plan_iterations:
    #    return Command(goto="reporter")
    messages = prepare_prompt("planner",configurable, messages)
    full_response = ""
    response = plan_llm.invoke(messages)
    full_response = response.model_dump_json(indent=4, exclude_none=True)
    plan_response = json.loads(full_response)
    new_plan = Plan.model_validate(plan_response)
    return Command(
        update={
            "messages": [AIMessage(content=full_response, name="planner")],
            "current_plan": new_plan,
        },
        goto="research_coordinator",
    )

In [12]:
def research_coordinator_node(state: State):
    print("Nothing")
    """Research team node that collaborates on tasks."""
    pass

In [13]:
from langgraph.checkpoint.memory import MemorySaver
memory = MemorySaver()

graph = StateGraph(State)

# Add nodes
graph.add_node("planner", planner_node)
graph.add_node("research_coordinator", research_coordinator_node)
graph.set_entry_point("planner")
graph.add_edge("research_coordinator", END)  # Option in the future, to add another step and filter the documents retrieved using rerhank before writing the report

agent = graph.compile(checkpointer=memory)

In [14]:
events = list(agent.stream(input=initial_state, config=config, stream_mode="values"))

Nothing


In [15]:
snapshot = agent.get_state(config)
{k for k, v in snapshot.values.items()}

{'current_plan', 'messages'}

In [16]:
import pprint
pprint.pprint(snapshot.values['messages'][1].content)

('{\n'
 '    "thought": "We will use the Washington EV registration SQLite database '
 'to quantify how EV MSRP evolved over time. First, discover the schema to '
 'identify the correct table(s) and fields (MSRP, model/registration year, EV '
 'type, make/model). Then, perform robust data-quality checks, filter '
 'outliers, and compute nominal MSRP trends by multiple lenses (model year vs '
 'registration year, BEV vs PHEV, Tesla vs non-Tesla, and top models), '
 'including averages, medians, and YoY changes. If a CPI table exists, compute '
 'inflation-adjusted trends; otherwise, report nominal results with '
 'limitations. Re-plan queries after seeing actual column names.",\n'
 '    "title": "Plan to analyze how EV MSRP in Washington changed over the '
 'years",\n'
 '    "steps": [\n'
 '        {\n'
 '            "title": "Inventory the database and identify relevant schema",\n'
 '            "description": "- Tool: list_tables to enumerate all tables.\\n- '
 'Tool: describe_table f

In [17]:
async def researcher_node(state: State, config: RunnableConfig):
    configurable = Configuration.from_runnable_config(config)
    messages = state.get("messages")
    plan = state.get("current_plan")
    plan_text = plan.model_dump_json(indent=2, exclude_none=True) if isinstance(plan, Plan) else str(plan)
    prompt = prepare_prompt("researcher", configurable, messages, plan=plan_text)
    llm = adv_llm.bind_tools(configurable.mcp_settings)
    ai_message = llm.invoke(prompt)
    new_messages = [ai_message]
    observations = []
    tool_map = {t.name: t for t in configurable.mcp_settings}
    for tc in ai_message.additional_kwargs.get("tool_calls", []):
        name = tc["function"]["name"]
        args = json.loads(tc["function"].get("arguments", "{}"))
        tool = tool_map.get(name)
        if tool:
            result = await tool.arun(args)
            tool_result = ToolMessage(content=json.dumps(result, indent=2), name=name, tool_call_id=tc["id"])
            new_messages.append(tool_result)
    return Command(update={"messages": new_messages})

In [18]:
from langgraph.checkpoint.memory import MemorySaver
memory = MemorySaver()

graph = StateGraph(State)

# Add nodes
graph.add_node("planner", planner_node)
graph.add_node("research_coordinator", researcher_node)
graph.set_entry_point("planner")
graph.add_edge("research_coordinator", END)  # Option in the future, to add another step and filter the documents retrieved using rerhank before writing the report

agent = graph.compile(checkpointer=memory)

events = []
async for s in agent.astream(input=initial_state, config=config, stream_mode="values"):
    message = s["messages"][-1]
    events.append(message)

In [19]:
snapshot = agent.get_state(config)
{k for k, v in snapshot.values.items()}

{'current_plan', 'messages'}

In [20]:
async def research_planner_node(state: State, config: RunnableConfig):
    configurable = Configuration.from_runnable_config(config)
    messages = state.get("messages")
    step_iteration = state.get("step_iteration") + 1
    plan = state.get("current_plan")
    context = state.get("observations", [])
    plan_text = plan.model_dump_json(indent=2, exclude_none=True) if isinstance(plan, Plan) else str(plan)
    if step_iteration > configurable.max_step_num:
        return Command(goto="reporter")
    prompt = prepare_prompt("research_planner", configurable, messages, plan=plan_text)
    prompt = prompt[0]["content"]
    response = mini_llm.invoke(prompt)
    full_response = response.model_dump_json(indent=4, exclude_none=True)
    next_step = json.loads(response.content)
    print(step_iteration)
    return Command(update={"messages": [AIMessage(content=full_response, name="research_planner")], "step_iteration": step_iteration,},
        goto=next_step['next_step'],)

In [21]:
def reporter_node(state: State):
    print("Nothing")
    """Research team node that collaborates on tasks."""
    pass

In [22]:
question = "How did the MSRP of EVs in Washington State change over the years?"

initial_state = {
        #Setting up State
        "messages": [{"role": "user", "content": question}],
        "step_iteration": 0,
    }

config = {
    #runtime configuration or variables
        "configurable": {
            "thread_id": "default",
            "max_plan_iterations": 1,
            "max_step_num": 8,
            "mcp_settings": tools
        },
        "recursion_limit": 100,
    }

In [23]:
from langgraph.checkpoint.memory import MemorySaver
memory = MemorySaver()

graph = StateGraph(State)

# Add nodes
graph.add_node("planner", planner_node)
graph.add_node("research_coordinator", researcher_node)
graph.add_node("research_planner", research_planner_node)
graph.add_node("reporter", reporter_node)
graph.set_entry_point("planner")
graph.add_edge("research_coordinator", "research_planner")
graph.add_edge("reporter", END)  # Option in the future, to add another step and filter the documents retrieved using rerhank before writing the report

agent = graph.compile(checkpointer=memory)

events = []
async for s in agent.astream(input=initial_state, config=config, stream_mode="values"):
    message = s["messages"][-1]
    events.append(message)

1
2
3
4
5
6
7
8
Nothing


In [24]:
snapshot = agent.get_state(config)
{k for k, v in snapshot.values.items()}

{'current_plan', 'messages', 'step_iteration'}

In [27]:
def reporter_node(state: State):
    configurable = Configuration.from_runnable_config(config)
    messages = state.get("messages")
    prompt = prepare_prompt("reporter", configurable, messages, plan=None)
    response = adv_llm.invoke(prompt)
    return Command(update={
            "final_report": response
        },)

In [28]:
from langgraph.checkpoint.memory import MemorySaver
memory = MemorySaver()

graph = StateGraph(State)

# Add nodes
graph.add_node("planner", planner_node)
graph.add_node("research_coordinator", researcher_node)
graph.add_node("research_planner", research_planner_node)
graph.add_node("reporter", reporter_node)
graph.set_entry_point("planner")
graph.add_edge("research_coordinator", "research_planner")
graph.add_edge("reporter", END)  # Option in the future, to add another step and filter the documents retrieved using rerhank before writing the report

agent = graph.compile(checkpointer=memory)

events = []
async for s in agent.astream(input=initial_state, config=config, stream_mode="values"):
    message = s["messages"][-1]
    events.append(message)

1
2
3
4
5
6
7
8


In [29]:
snapshot = agent.get_state(config)
{k for k, v in snapshot.values.items()}

{'current_plan', 'final_report', 'messages', 'step_iteration'}

In [30]:
print(snapshot[0]['final_report'].content)

Summary
- We could not access the Washington EV database to compute empirical MSRP trends. Every attempt to inventory the database failed with the error “unable to open database file,” so no tables, schemas, or data could be read.
- Because no data were accessible, we cannot quantify how EV MSRP in Washington changed over time from this dataset yet.
- Below, I document what was attempted, what is needed to proceed, the exact analyses planned (means/medians by year, EV-type splits, make/model mix), and caveats. Once access is restored or a CSV is provided, I can produce the trend tables and a concise narrative (e.g., % change in median MSRP from earliest to latest year).

What was attempted
- Tool calls to list tables from the SQLite database repeatedly returned: Error listing tables: unable to open database file.
- Without table access, I couldn’t discover schemas or run any SQL to compute year-by-year MSRP summaries.

What is needed to proceed
Please provide one of the following so I 